# Presque singulière

| Module | Durée | Prérequis |
|---|---|---|
| M4 – Inverse, déterminant, rang | 75 min | M2 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une petite librairie scanne le code-barres des livres d'occasion avec la caméra d'une tablette. Quand la tablette bouge, la photo est **floue** : chaque pixel est mélangé avec ses voisins, et le lecteur ne reconnaît plus les barres.

Yanis, stagiaire, a une idée : « Le flou est une opération linéaire. On connaît sa matrice $B$. Il suffit de l'inverser ! »

**Les questions : l'idée de Yanis fonctionne-t-elle ? Sinon, pourquoi, et que faire ?**

**Activité** : Listez les informations nécessaires : que faut-il savoir sur le flou ? sur le capteur de la tablette ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Le capteur ajoute un petit bruit, d'environ 1 % de l'intensité. Sur un code-barres de 80 pixels, combien de pixels l'inversion du flou va-t-elle mal reconstituer ?

In [ ]:
pari_pixels_faux = 2     # sur 80 (modifiez)

## ③ Abstraire

On représente une ligne du code-barres par un vecteur $\mathbf{x}$ de 80 valeurs (0 = blanc, 1 = noir). Le flou est une matrice $B$ : chaque pixel flou est une moyenne pondérée de ses voisins. Le capteur ajoute un bruit $\boldsymbol{\varepsilon}$ :

$$
\mathbf{y} = B\,\mathbf{x} + \boldsymbol{\varepsilon}
$$

L'idée de Yanis est de calculer $B^{-1}\mathbf{y}$. Or :

$$
B^{-1}\mathbf{y} = \mathbf{x} + B^{-1}\boldsymbol{\varepsilon}
$$

![Schéma : le code-barres net passe par le flou B, reçoit le bruit du capteur, puis l'inversion naïve renvoie x plus B inverse fois le bruit](../../assets/schemas/schema-flou.svg)

Le flou est parfaitement annulé. Toute la question est donc la taille de $B^{-1}\boldsymbol{\varepsilon}$, autrement dit à quel point $B^{-1}$ amplifie un petit vecteur :

$$
\lVert B^{-1}\boldsymbol{\varepsilon} \rVert \le \lVert B^{-1} \rVert \, \lVert \boldsymbol{\varepsilon} \rVert,
\qquad
\operatorname{cond}(B) = \lVert B \rVert \, \lVert B^{-1} \rVert
$$

C'est le **conditionnement** du notebook 2.4, vu cette fois à travers la matrice inverse.

**Un piège : le déterminant.** On pourrait croire qu'un déterminant proche de 0 signale une matrice « presque singulière ». C'est faux : $\det(0{,}1\,I_{10}) = 10^{-10}$, alors que diviser par 0,1 ne pose aucun problème. Le déterminant mesure un changement de **volume**, pas une **instabilité**.

## ④ Calculer

### Le code-barres et le flou

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import toeplitz

n = 80
rng = np.random.default_rng(3)
x = np.repeat((rng.random(n // 2) < 0.5).astype(float), 2)     # barres de 2 pixels

def matrice_flou(n, etalement):
    # Chaque pixel devient une moyenne de ses voisins, pondérée par une cloche de largeur « etalement »
    noyau = np.exp(-0.5 * (np.arange(n) / etalement) ** 2)
    B = toeplitz(noyau)
    return B / B.sum(axis=1, keepdims=True)

B = matrice_flou(n, 1.5)
bruit = 0.01 * np.random.default_rng(5).normal(size=n)
y = B @ x + bruit

print(f"det(B)  = {np.linalg.det(B):.2e}")
print(f"cond(B) = {np.linalg.cond(B):.2e}")
print(f"det(0,1 I) = {np.linalg.det(0.1 * np.eye(10)):.0e}   cond(0,1 I) = {np.linalg.cond(0.1 * np.eye(10)):.0f}")

### L'idée de Yanis

In [ ]:
x_sans_bruit = np.linalg.solve(B, B @ x)       # si le capteur était parfait
x_naif = np.linalg.solve(B, y)                 # avec 1 % de bruit

def pixels_faux(estimation):
    return int(np.sum((estimation > 0.5) != (x > 0.5)))

print("Sans bruit : pixels faux =", pixels_faux(x_sans_bruit))
print("Avec bruit : pixels faux =", pixels_faux(x_naif), "  valeurs entre", round(x_naif.min()), "et", round(x_naif.max()))
print("Photo floue lue telle quelle : pixels faux =", pixels_faux(y))

### Dompter le bruit : la régularisation

Au lieu d'exiger $B\mathbf{x} = \mathbf{y}$ exactement, on cherche un $\mathbf{x}$ qui explique à peu près la photo **sans prendre de valeurs extravagantes**. Le compromis est réglé par un nombre $\lambda > 0$. Le module M6 montrera d'où vient cette recette ; ici, on l'utilise comme un outil :

$$
(B^{\top}B + \lambda I)\,\mathbf{x} = B^{\top}\mathbf{y}
$$

Cette méthode s'appelle la **régularisation de Tikhonov**. La matrice $B^{\top}B + \lambda I$ est bien mieux conditionnée que $B$.

In [ ]:
def tikhonov(B, y, lam):
    return np.linalg.solve(B.T @ B + lam * np.eye(B.shape[1]), B.T @ y)

x_reg = tikhonov(B, y, 1e-3)
print(f"cond(B^T B + 0,001 I) = {np.linalg.cond(B.T @ B + 1e-3 * np.eye(n)):.1e}")
print("Pixels faux avec régularisation :", pixels_faux(x_reg))

fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
for ax, signal, titre in zip(axes, [x, y, x_naif, x_reg],
                             ["Code-barres net", "Photo floue et bruitée", "Inversion naïve : B⁻¹ y", "Régularisation (λ = 0,001)"]):
    ax.step(range(n), signal, where="mid", color=LIOUAI["navy"], lw=1.4)
    ax.fill_between(range(n), signal, step="mid", color=LIOUAI["vert"], alpha=0.35)
    ax.set_title(titre, fontsize=10, loc="left"); ax.set_yticks([])
axes[2].set_ylim(-3, 4)
axes[-1].set_xlabel("pixel")
plt.tight_layout(); plt.show()

### Choisir λ

In [ ]:
lambdas = np.logspace(-7, 1, 33)
faux = [pixels_faux(tikhonov(B, y, lam)) for lam in lambdas]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.semilogx(lambdas, faux, "o-")
ax.axhline(pixels_faux(y), ls="--", color=LIOUAI["gris"], label="photo floue lue telle quelle")
ax.set_xlabel("λ"); ax.set_ylabel("pixels faux (sur 80)"); ax.legend()
ax.set_title("Trop peu de régularisation : le bruit explose ; trop : on refloute")
plt.tight_layout(); plt.show()

### Inverser ou résoudre ?

`np.linalg.inv(B) @ y` et `np.linalg.solve(B, y)` donnent mathématiquement le même résultat. En pratique, calculer l'inverse est plus lent et souvent moins précis : on le réserve aux cas où l'on a besoin de la matrice $B^{-1}$ elle-même.

In [ ]:
import time
grand = np.random.default_rng(0).normal(size=(1500, 1500)) + 40 * np.eye(1500)
second_membre = np.ones(1500)
debut = time.perf_counter(); s1 = np.linalg.solve(grand, second_membre); t1 = time.perf_counter() - debut
debut = time.perf_counter(); s2 = np.linalg.inv(grand) @ second_membre; t2 = time.perf_counter() - debut
print(f"solve : {t1:.3f} s, résidu {np.linalg.norm(grand @ s1 - second_membre):.1e}")
print(f"inv   : {t2:.3f} s, résidu {np.linalg.norm(grand @ s2 - second_membre):.1e}")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_pixels_faux} pixels faux ; inversion naïve : {pixels_faux(x_naif)} sur 80")

**Activité** :

1. Répondez à Yanis : son idée est-elle fausse, ou seulement inapplicable avec un vrai capteur ? Appuyez-vous sur les deux résultats « sans bruit » et « avec bruit ».
2. $\det(B)$ est minuscule, et $\det(0{,}1\,I)$ aussi. Pourquoi l'un est-il un vrai problème et pas l'autre ?
3. Sur la courbe en fonction de $\lambda$, décrivez les trois zones. Que se passe-t-il pour un $\lambda$ très grand ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Les limites du calcul.** Listez les sources d'erreur de cette chaîne : modèle du flou (connaît-on vraiment l'étalement ?), bruit du capteur, arrondis de la machine. Laquelle domine ?
- **Faisabilité.** Recommencez avec un flou plus fort (étalement 2,5). Le conditionnement passe à environ $10^{13}$. La régularisation suffit-elle encore ? À partir de quand vaut-il mieux reprendre la photo ?
- **Décider.** Écrivez une fonction `deflouer(y, B, lambdas)` qui choisit automatiquement $\lambda$. Sans connaître le vrai code-barres, quel critère utiliser ? Une piste : choisir le $\lambda$ pour lequel $\lVert B\hat{\mathbf{x}} - \mathbf{y} \rVert$ est de l'ordre du bruit attendu (environ $0{,}01\sqrt{80}$).

In [ ]:
def deflouer(y, B, lambdas, niveau_bruit=0.01):
    # À compléter : renvoyer (meilleur lambda, estimation)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def deflouer(y, B, lambdas, niveau_bruit=0.01):
    # Principe de l'écart de Morozov : le résidu doit avoir la taille du bruit, pas moins
    cible = niveau_bruit * np.sqrt(len(y))
    for lam in sorted(lambdas, reverse=True):             # du plus régularisé au moins régularisé
        estimation = tikhonov(B, y, lam)
        if np.linalg.norm(B @ estimation - y) <= cible:
            return lam, estimation
    return min(lambdas), tikhonov(B, y, min(lambdas))

lam, estimation = deflouer(y, B, lambdas)
print(f"λ choisi sans connaître x : {lam:.1e} ; pixels faux : {pixels_faux(estimation)}")

B_fort = matrice_flou(n, 2.5)
y_fort = B_fort @ x + bruit
lam_f, est_f = deflouer(y_fort, B_fort, lambdas)
print(f"Flou fort : cond = {np.linalg.cond(B_fort):.0e}, λ choisi {lam_f:.1e}, pixels faux : {pixels_faux(est_f)}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Lister les sources d'erreur liées aux limites du calcul": 1,
    "Sentir si le résultat est plausible pour le contexte": 1,
    "Sentir la fiabilité d'un modèle": 1,
    "Évaluer la faisabilité d'obtenir une réponse utile": 1,
    "Décider si les résultats suffisent pour avancer ou abandonner": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "4.3-presque-singuliere", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")